In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.9/776.9 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 36.5 MB/s eta 0:00:00
ERROR: unknown command "instlal" - maybe you meant "install"


In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.optimization import get_cosine_with_hard_restarts_schedule_with_warmup

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=32
    weight_decay=1e-6
    max_len = 60
    fold = 5

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_test.csv', encoding = 'utf-8')
df_train_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/데이터전처리/df_train_pre.csv', encoding = 'utf-8')
df_dev_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/데이터전처리/df_dev_pre.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/데이터전처리/df_test_pre.csv', encoding = 'utf-8')

df_train_pre.dropna(inplace = True)
df_dev_pre.dropna(inplace = True)
df_test_pre.dropna(inplace = True)

df_train_pre.reset_index(drop = True, inplace = True)
df_dev_pre.reset_index(drop = True, inplace = True)

cuda


In [ ]:
from tokenizers.processors import TemplateProcessing

class CustomDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

model_name = "beomi/KcELECTRA-base-v2022"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/450k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size)

dir = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/kcelectra_fold0.pt'
torch.save(model.state_dict(), dir)

kf = KFold(n_splits=args.fold, shuffle=True, random_state=42)

df = pd.concat([df_train_pre, df_dev_pre])

# KFold 교차 검증
fold = 0
for fold, (train_index, valid_index) in enumerate(kf.split(df)):
    print(f"Fold {fold + 1}/{args.fold}")

    model.load_state_dict(torch.load(dir))
    model.to(device)

    df_train = df.iloc[train_index].reset_index(drop=True)
    df_valid = df.iloc[valid_index].reset_index(drop=True)

    train_dataset = CustomDataset(df_train, 'input', 'output', tokenizer, args.max_len)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)

    valid_dataset = CustomDataset(df_valid, 'input', 'output', tokenizer, args.max_len)
    valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=args.start_lr)
    scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(optimizer,
                                                                num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                                                num_training_steps = len(train_loader)*args.epochs)

    stop_val_f1 = 0
    stop_count = 0

    for epoch in range(args.epochs):
        if stop_count == 4:
            break
        model.train()
        train_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(train_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            outputs = model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()
            scheduler.step()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / len(pbar), 4)))

        train_loss = train_loss / len(train_loader)
        train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='micro')
        train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        model.eval()
        valid_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(valid_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            outputs = model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            valid_loss += loss.item()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / len(pbar), 4)))
        valid_loss = valid_loss / len(valid_loader)
        valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='micro')
        valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        print(f'Fold {fold + 1}, Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
        print(f'                     v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

        if valid_score > stop_val_f1:
            default_weight_path = path + f'kcelectra_fold{fold + 1}_.pt'
            torch.save(model.state_dict(), default_weight_path)
            stop_val_f1 = valid_score
            stop_count = 0
        else:
            stop_count += 1

        torch.cuda.empty_cache()
        gc.collect()

pytorch_model.bin:   0%|          | 0.00/511M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.dense.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fold 1/5


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
[C_loss : 0.1572]: 100%|██████████| 116/116 [00:06<00:00, 16.91it/s]


Fold 1, Epoch : 1,    t_loss : 0.3811,   t_f1 : 0.8173,   t_acc : 0.8173

                     v_loss : 0.1572,   v_f1 : 0.9476,   v_acc : 0.9476



...

Fold 5, Epoch : 8,    t_loss : 0.0096,   t_f1 : 0.9977,   t_acc : 0.9977

                     v_loss : 0.2996,   v_f1 : 0.9443,   v_acc : 0.9443



[C_loss : 0.3017]: 100%|██████████| 116/116 [00:07<00:00, 15.33it/s]


Fold 5, Epoch : 9,    t_loss : 0.0077,   t_f1 : 0.9984,   t_acc : 0.9984

                     v_loss : 0.3017,   v_f1 : 0.947,   v_acc : 0.947



[C_loss : 0.3024]: 100%|██████████| 116/116 [00:06<00:00, 18.17it/s]


Fold 5, Epoch : 10,    t_loss : 0.0063,   t_f1 : 0.9987,   t_acc : 0.9987

                     v_loss : 0.3024,   v_f1 : 0.9476,   v_acc : 0.9476



In [ ]:
class TestDataset(Dataset):
    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

model_name = "beomi/KcELECTRA-base-v2022"
tokenizer = AutoTokenizer.from_pretrained(model_name)
test_dataset = TestDataset(df_test_pre, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

In [ ]:
model_paths = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/kcelectra_fold1_.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/kcelectra_fold2_.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/kcelectra_fold3_.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/kcelectra_fold4_.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/kcelectra_fold5_.pt']
models = []
for model_path in model_paths:
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()
    models.append(model)

res = []
with torch.no_grad():
    for _, data in enumerate(tqdm(test_loader)):
        ids = data[0].to(device)
        mask = data[1].to(device)

        outputs = [model(ids, mask)[0] for model in models]

        avg_output = sum(outputs) / len(models)
        avg_output = avg_output.cpu().numpy()

        res.extend(avg_output.argmax(axis=1).tolist())

df_test['output'] = res

100%|██████████| 65/65 [00:11<00:00,  5.84it/s]


In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",1
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,1
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/test27_kfold_kcelectra_base_new.jsonl')